AutoRAG Banner

## Notebook content

This notebook presents the AutoRAG steps: data preparation, experiment execution, leaderboard analysis of the generated RAG patterns, and querying the selected pattern.

### Contents 
This notebook contains the following parts:
- **[Setup](#Setup)**
- **[Prepare experiment data](#Prepare-experiment-data)**
- **[Process input documents](#Process-input-documents)**
- **[Run ai4rag experiment](#Run-ai4rag-experiment)**
- **[Review experiment results](#review-experiment-results)**
- **[Summary and next steps](#Summary-and-next-steps)**

## Setup

Install packages

In [1]:
!pip install boto3 | tail -n 1
!pip install --no-cache-dir git+https://github.com/LukaszCmielowski/pipelines-components.git@rhoai_autorag_data_processing_pipeline 2>&1 | tail -n 1
# !pip install "git+https://github.com/IBM/ai4rag.git@dev" | tail -n 1

Found existing installation: kfp-components 1.11.0
Uninstalling kfp-components-1.11.0:
  Successfully uninstalled kfp-components-1.11.0
  Created wheel for kfp-components: filename=kfp_components-1.11.0-py3-none-any.whl size=27127 sha256=49e5c79d770b3450c5821659e297e0e50111b0858a0e11eddbe4bc85f1ea2104
  Stored in directory: /private/var/folders/3x/80z00mdn0kl0r08j0zc2ssr40000gn/T/pip-ephem-wheel-cache-_r44yenh/wheels/62/7f/40/c65ae1bd9a71463506539fcf88d59a7aa6927696e1d20dc2cb
Successfully built kfp-components

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Import required libraries

In [1]:
import re
import os
import json
import yaml
import boto3
import logging
import urllib.request
from pathlib import Path
from types import SimpleNamespace

import warnings
warnings.filterwarnings("ignore")

import ibm_boto3
from ibm_botocore.client import Config
from langchain_core.documents import Document

for logger_name in (
    "Test Data Loader component logger",
    "Document Loader component logger",
    "Text Extraction component logger",
):
    logging.getLogger(logger_name).propagate = False

📌 **Action**: Provide the credentials for your S3 instance if they are not already set in the notebook environment.

In [3]:
AWS_ACCESS_KEY_ID = ""
AWS_SECRET_ACCESS_KEY = ""
AWS_S3_ENDPOINT = ""
AWS_DEFAULT_REGION = ""

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID or os.environ.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY or os.environ.get("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_S3_ENDPOINT"] = AWS_S3_ENDPOINT or os.environ.get("AWS_S3_ENDPOINT")
os.environ["AWS_DEFAULT_REGION"] = AWS_DEFAULT_REGION or os.environ.get("AWS_DEFAULT_REGION")

📌 **Action**: Provide the bucket name where the experiment data will be stored.

> 🔖 **Note**: Bucket must already exists.

In [4]:
BUCKET_NAME = ""

BUCKET_NAME = BUCKET_NAME or os.environ.get("BUCKET_NAME")
if not BUCKET_NAME:
    raise Exception("BUCKET_NAME must be provided")

## Prepare experiment data

#### Initialize S3 client

In [5]:
session = boto3.session.Session(
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)
s3_client = session.client(
    service_name='s3',
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
)

#### Upload documents
For the needs of this notebook we are using IBM financial reports available here: https://www.ibm.com/investor/financial-reporting

In [6]:
documents_urls = [
    "https://www.ibm.com/downloads/documents/us-en/12bb2f913a3ba1a2",
    "https://www.ibm.com/downloads/documents/us-en/131cf8a39db327fd",
    "https://www.ibm.com/downloads/documents/us-en/131cf87ab633199f",
    "https://www.ibm.com/downloads/documents/us-en/1550f7eea8c0ded6",
    "https://www.ibm.com/downloads/documents/us-en/10a9980400afd114",
    "https://www.ibm.com/downloads/documents/us-en/10a9980468afdf4c",
    "https://www.ibm.com/downloads/documents/us-en/10a9980400afd11c",
    "https://www.ibm.com/downloads/documents/us-en/11ed3283ae56ec71"
]

for url in documents_urls:
    with urllib.request.urlopen(url) as response:
        content = response.read()
        content_disposition = response.headers.get("Content-Disposition")
        filename = re.findall('filename="(.+)"', content_disposition)[0]
        s3_client.put_object(Bucket=BUCKET_NAME, Key=f"documents/{filename}", Body=content)

#### Upload benchmark dataset

In [7]:
benchmark = [
    {
        "question": "What was IBM's revenue in the first quarter of 2024?",
        "correct_answers": [
            "Revenue of $14.5 billion, up 1 percent, up 3 percent at constant currency."
        ],
        "correct_answer_document_ids": [
            "ibm-1q25-earnings-press-release.pdf"
        ]
    },
    {
        "question": "What did IBM announce regarding HashiCorp in first quarter 2024?",
        "correct_answers": [
            "IBM announced its intent to acquire HashiCorp, Inc. for $35 per share in cash, representing an enterprise value of $6.4 billion. The transaction was expected to close by the end of 2024."
        ],
        "correct_answer_document_ids": [
            "ibm-1q25-earnings-press-release.pdf"
        ]
    },
        {
        "question": "How much did IBM invest in acquisitions in the first quarter of 2025, including HashiCorp?",
        "correct_answers": [
            "The company invested $7.1 billion in acquisitions, including the acquisition of HashiCorp."
        ],
        "correct_answer_document_ids": [
            "ibm-1q25-earnings-press-release.pdf"
        ]
    },
    {
        "question": "What quarterly dividend did the IBM board approve in July 2025?",
        "correct_answers": [
            "On July 23, 2025, the IBM board of directors approved a regular quarterly cash dividend of $1.68 per common share, to stockholders of record on August 8, 2025."
        ],
        "correct_answer_document_ids": [
            "ibm-2q25-earnings-press-release.pdf"
        ]
    },
    {
        "question": "What was IBM's third quarter 2024 revenue and Software growth?",
        "correct_answers": [
            "Revenue of $15.0 billion, up 1 percent, up 2 percent at constant currency. Software revenue up 10 percent."
        ],
        "correct_answer_document_ids": [
            "ibm-3q24-earnings-press-release.pdf"
        ]
    },
]

res = s3_client.put_object(Bucket=BUCKET_NAME, Key="benchmark.json", Body=json.dumps(benchmark))

Look up bucket contents

In [8]:
res = s3_client.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix="",
).get("Contents", [])

print("Bucket contents:")
for r in res:
    print(r["Key"])

Bucket contents:
benchmark.json
documents/4q25-press-release.pdf
documents/ibm-1q24-earnings-press-release.pdf
documents/ibm-1q25-earnings-press-release.pdf
documents/ibm-2q24-earnings-press-release.pdf
documents/ibm-2q25-earnings-press-release.pdf
documents/ibm-3q-25-press-release.pdf
documents/ibm-3q24-earnings-press-release.pdf
documents/ibm-4q24-earnings-press-release.pdf


## Process input documents

The data processing flow prepares input for the experiment in three steps. Each step runs as a standalone component (via `python_func`) with artifact paths under `step_outputs/`. Ensure `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_S3_ENDPOINT`, and `AWS_DEFAULT_REGION` are set in your environment before running.

| Step | Component | Purpose |
|------|-----------|---------|
| 1 | **Test data loader** | Download the benchmark JSON (questions and ground truth) from S3. |
| 2 | **Documents sampling** | List documents in the bucket, prioritize benchmark-referenced docs, apply a size cap, and write a YAML manifest (no content download). |
| 3 | **Text extraction** | Download the listed documents from S3 and extract text to Markdown using Docling. |

In [9]:
from kfp_components.components.data_processing.autorag.test_data_loader.component import test_data_loader
from kfp_components.components.data_processing.autorag.documents_sampling.component import documents_sampling
from kfp_components.components.data_processing.autorag.text_extraction.component import text_extraction

step_output_dir = Path("./step_outputs")
step_output_dir.mkdir(parents=True, exist_ok=True)

test_data_bucket_name = BUCKET_NAME
test_data_key = "benchmark.json"
input_data_bucket_name = BUCKET_NAME
input_data_key = "documents"
sampling_config = {}

#### Step 1: Test data loader

Downloads the benchmark (test data) JSON file from the configured S3 bucket and key. The file must be a JSON containing a list of items with `question`, `correct_answers`, and `correct_answer_document_ids`.

In [10]:
test_data_out = SimpleNamespace(path=str(step_output_dir / "test_data.json"))

test_data_loader.python_func(
    test_data_bucket_name=test_data_bucket_name,
    test_data_path=test_data_key,
    test_data=test_data_out,
)

output_path = Path(test_data_out.path)
with output_path.open("r", encoding="utf-8") as f:
    test_data = json.load(f)

print(json.dumps(test_data, indent=4, ensure_ascii=False))

Fetching test data from S3: bucket=autorag-datasets, path=benchmark.json
Starting download to step_outputs/test_data.json
Download completed successfully
[
    {
        "question": "What was IBM's revenue in the first quarter of 2024?",
        "correct_answers": [
            "Revenue of $14.5 billion, up 1 percent, up 3 percent at constant currency."
        ],
        "correct_answer_document_ids": [
            "ibm-1q25-earnings-press-release.pdf"
        ]
    },
    {
        "question": "What did IBM announce regarding HashiCorp in first quarter 2024?",
        "correct_answers": [
            "IBM announced its intent to acquire HashiCorp, Inc. for $35 per share in cash, representing an enterprise value of $6.4 billion. The transaction was expected to close by the end of 2024."
        ],
        "correct_answer_document_ids": [
            "ibm-1q25-earnings-press-release.pdf"
        ]
    },
    {
        "question": "How much did IBM invest in acquisitions in the first qu

#### Step 2: Documents sampling

Lists objects in the S3 input bucket, filters by supported extensions (e.g. `.pdf`, `.docx`, `.pptx`, `.md`, `.html`, `.txt`), and builds a sampled set: documents referenced in the benchmark (from Step 1) are prioritized; then others are added until a configurable size limit (1 GB as default) is reached. **This step does not download document contents.** It writes a YAML manifest, `sampled_documents_descriptor.yaml`, containing bucket, prefix, and the list of selected object keys and sizes. That manifest is the input for the text extraction step.

In [11]:
test_data_in = SimpleNamespace(path=str(step_output_dir / "test_data.json"))
sampled_documents_out = SimpleNamespace(path=str(step_output_dir / "sampled_documents"))

documents_sampling.python_func(
    input_data_bucket_name=input_data_bucket_name,
    input_data_path=input_data_key,
    test_data=test_data_in,
    sampling_config=sampling_config,
    sampled_documents=sampled_documents_out,
)

descriptor_path = step_output_dir / "sampled_documents" / "sampled_documents_descriptor.yaml"
with open(descriptor_path) as f:
    descriptor = yaml.safe_load(f)

print(json.dumps(descriptor, indent=4, ensure_ascii=False))

Sampled documents descriptor content {'bucket': 'autorag-datasets', 'prefix': 'documents', 'documents': [{'key': 'documents/4q25-press-release.pdf', 'size_bytes': 278516}, {'key': 'documents/ibm-1q24-earnings-press-release.pdf', 'size_bytes': 211288}, {'key': 'documents/ibm-1q25-earnings-press-release.pdf', 'size_bytes': 230841}, {'key': 'documents/ibm-2q24-earnings-press-release.pdf', 'size_bytes': 278765}, {'key': 'documents/ibm-2q25-earnings-press-release.pdf', 'size_bytes': 280313}, {'key': 'documents/ibm-3q-25-press-release.pdf', 'size_bytes': 275585}, {'key': 'documents/ibm-3q24-earnings-press-release.pdf', 'size_bytes': 280378}, {'key': 'documents/ibm-4q24-earnings-press-release.pdf', 'size_bytes': 290324}], 'total_size_bytes': 2126010, 'count': 8}
Sampled documents descriptor written to step_outputs/sampled_documents/sampled_documents_descriptor.yaml
{
    "bucket": "autorag-datasets",
    "prefix": "documents",
    "documents": [
        {
            "key": "documents/4q25-pr

#### Step 3: Text extraction

Reads the `sampled_documents_descriptor.yaml` produced by Step 2, downloads each listed document from S3 into a temporary directory, and runs **Docling** to extract text. Output is one Markdown file per document (e.g. `document_0.md`, `document_1.md`) written to the artifact output path. These files are the final text corpus for the experiment.

In [12]:
sampled_descriptor_in = SimpleNamespace(path=str(step_output_dir / "sampled_documents"))
extracted_text_out = SimpleNamespace(path=str(step_output_dir / "extracted_text"))

text_extraction.python_func(
    sampled_documents_descriptor=sampled_descriptor_in,
    extracted_text=extracted_text_out,
)

Starting text extraction for 8 documents.
Text extraction completed. Total processed: 8, Errors: 0.


Load the extracted Markdown files from Step 3 into LangChain `Document` objects.
> 🔖 **Note:** Document metadata must contain `document_id` key with the name of the document as it was refered in the benchmark data json file.

In [13]:
paths = list(Path("step_outputs/extracted_text").glob("*.md"))
documents = [
    Document(
        page_content=p.read_text(encoding="utf-8", errors="replace"),
        metadata={"document_id": p.stem},
    )
    for p in sorted(paths)
]
documents[0].page_content[:500]

n = 3
print(f"First {n} documents:")
for doc in documents[:n]:
    print("=" * 100)
    print(doc.metadata)
    print(doc.page_content[:800])


First 3 documents:
{'document_id': '4q25-press-release.pdf'}
## IBM RELEASES FOURTH-QUARTER RESULTS

Strong, broad-based performance, led by double-digit Software and Infrastructure growth; Double-digit growth in full-year profit and free cash flow

ARMONK, N.Y., January 28, 2026 . . . IBM (NYSE: IBM) today announced fourth-quarter 2025 earnings results.

'In the fourth quarter, we delivered strong revenue growth, with double -digit Software performance. Additionally, Infrastructure continued its double-digit revenue growth with the robust adoption of the next generation of our mainframe platform. Our generative AI book of business now stands at more than $12.5 billion. This capped a strong 2025 for IBM where we exceeded expectations for revenue, profit and free cash flow," said Arvind Krishna, IBM chairman, president and chief executive officer. "W
{'document_id': 'ibm-1q24-earnings-press-release.pdf'}
## IBM RELEASES FIRST-QUARTER RESULTS

## Accelerated Software revenue growth; Stro

## Run ai4rag experiment

In [ ]:
import os
from llama_stack_client import LlamaStackClient

client = LlamaStackClient(base_url=os.getenv("LLAMA_STACK_CLIENT_BASE_URL"), api_key=os.getenv("LLAMA_STACK_CLIENT_API_KEY"))

client.models.list()

In [ ]:
from ai4rag.core.hpo.gam_opt import GAMOptSettings
from ai4rag.search_space.src.parameter import Parameter
from ai4rag.search_space.src.search_space import AI4RAGSearchSpace
from ai4rag.rag.embedding.llama_stack import LSEmbeddingModel
from ai4rag.rag.foundation_models.llama_stack import LSFoundationModel


# Configure optimizer
optimizer_settings = GAMOptSettings(max_evals=10, n_random_nodes=4)

foundation_models = [LSFoundationModel(model_id="<PROVIDE SELECTED MODEL ID>", client=client)]
embedding_models = [LSEmbeddingModel(model_id="<PROVIDE SELECTED MODEL ID>", client=client, params={})]

# Edit configurations of search space
search_space = AI4RAGSearchSpace(
    params=[
        Parameter(name="foundation_model", param_type="C", values=foundation_models),
        Parameter(name="embedding_model", param_type="C", values=embedding_models),
    ]
)

In [ ]:
import pandas as pd

from ai4rag.core.experiment.experiment import AI4RAGExperiment
from ai4rag.utils.event_handler import LocalEventHandler
from ai4rag.core.hpo.gam_opt import GAMOptimizer


experiment = AI4RAGExperiment(
    client=client,
    documents=documents,
    benchmark_data=pd.DataFrame(test_data),
    search_space=search_space,
    optimizer_settings=optimizer_settings,
    optimization_metric="faithfulness",
    event_handler=LocalEventHandler(output_path=Path(".").absolute() / "ai4rag_results"),
    vector_store_type="ls_milvus",
)

experiment.search(optimizer=GAMOptimizer)

In [ ]:
best_eval = experiment.results.get_best_evaluations(k=1)[0]
best_eval

#### Format evaluation result for review

Display the evaluation result in a clean, tabular format showing scores, parameters, and metadata.

In [ ]:
def format_evaluation_result(eval_result):
    """
    Display EvaluationResult in a clean, readable format for notebook review.
    
    Args:
        eval_result: EvaluationResult instance
    """
    from IPython.display import display
    
    # Overall summary
    print(f"\n{'='*80}")
    print(f"Pattern: {eval_result.pattern_name}")
    print(f"Final Score: {eval_result.final_score:.4f}")
    print(f"Execution Time: {eval_result.execution_time:.2f}s")
    print(f"{'='*80}\n")
    
    # Aggregate scores table
    if "scores" in eval_result.scores:
        scores_data = []
        for metric, values in eval_result.scores["scores"].items():
            scores_data.append({
                "Metric": metric,
                "Mean": f"{values['mean']:.4f}",
                "CI Low": f"{values['ci_low']:.4f}",
                "CI High": f"{values['ci_high']:.4f}"
            })
        
        print("Evaluation Scores:")
        display(pd.DataFrame(scores_data))
        print()
    
    # Parameters table
    all_params = {**eval_result.indexing_params, **eval_result.rag_params}
    params_data = []
    for key, value in all_params.items():
        # Convert objects to readable strings (avoid memory addresses)
        if hasattr(value, '__class__'):
            if hasattr(value, 'model_id'):
                value_str = f"{value.__class__.__name__}({value.model_id})"
            else:
                value_str = value.__class__.__name__
        else:
            value_str = str(value)
        
        params_data.append({"Parameter": key, "Value": value_str})
    
    print("Configuration Parameters:")
    display(pd.DataFrame(params_data))

# Display the best evaluation
format_evaluation_result(best_eval)

## Review experiment results

Query the RAG pattern and display the answer along with grounding documents.

In [ ]:
def query_rag_pattern(rag_pattern, question, **retrieval_kwargs):
    """
    Query the RAG pattern and display the answer with grounding documents.
    
    Args:
        rag_pattern: RAG pattern instance (from EvaluationResult.rag_pattern)
        question: Question to ask
        **retrieval_kwargs: Additional retrieval parameters (e.g., number_of_chunks)
    
    Returns:
        dict: Response containing answer, reference_documents, and question
    """
    print(f"\n{'='*80}")
    print(f"Question: {question}")
    print(f"{'='*80}\n")
    
    # Query the RAG pattern
    response = rag_pattern.generate(question, **retrieval_kwargs)
    
    # Display answer
    print("Answer:")
    print(f"{response['answer']}\n")
    
    # Display grounding documents
    print(f"{'='*80}")
    print(f"Grounding Documents ({len(response['reference_documents'])} retrieved):")
    print(f"{'='*80}\n")
    
    for idx, doc in enumerate(response['reference_documents'], 1):
        doc_id = doc.metadata.get('document_id', 'Unknown')
        content_preview = doc.page_content[:300].replace('\n', ' ')
        if len(doc.page_content) > 300:
            content_preview += "..."
        
        print(f"[{idx}] Document ID: {doc_id}")
        print(f"    Content: {content_preview}")
        print()
    
    return response


def interactive_rag_query(rag_pattern, **retrieval_kwargs):
    """
    Create an interactive widget to query the RAG pattern.
    
    Args:
        rag_pattern: RAG pattern instance (from EvaluationResult.rag_pattern)
        **retrieval_kwargs: Additional retrieval parameters (e.g., number_of_chunks)
    """
    try:
        from ipywidgets import Textarea, Button, VBox, Output
        from IPython.display import display, clear_output
    except ImportError:
        print("ipywidgets not available. Use query_rag_pattern() function instead.")
        return
    
    # Create widgets
    question_input = Textarea(
        value='',
        placeholder='Enter your question here...',
        description='Question:',
        layout={'width': '100%', 'height': '80px'}
    )
    
    submit_button = Button(
        description='Ask',
        button_style='primary',
        tooltip='Submit question'
    )
    
    output = Output()
    
    def on_submit(b):
        with output:
            clear_output()
            question = question_input.value.strip()
            if question:
                query_rag_pattern(rag_pattern, question, **retrieval_kwargs)
            else:
                print("Please enter a question.")
    
    submit_button.on_click(on_submit)
    
    # Display widgets
    display(VBox([question_input, submit_button, output]))


# Example usage with a simple function call
print("Use query_rag_pattern() for direct queries or interactive_rag_query() for a widget interface.")

In [ ]:
# Example 1: Direct query
# query_rag_pattern(best_eval.rag_pattern, "What was IBM's revenue in Q1 2024?")

# Example 2: Interactive widget (uncomment to use)
# interactive_rag_query(best_eval.rag_pattern)

## Summary and next steps

**Summary:** This notebook set up the experiment data, processed it, ran the ai4rag experiment, displayed the RAG patterns leaderboard, and queried the selected pattern.

**Next steps:**
